In [ ]:
!pip install pydantic

In [ ]:
from openai import OpenAI
import gradio as gr
import pandas as pd
from pydantic import BaseModel, EmailStr
from typing import List
import tempfile

In [ ]:
# OpenAI client
import os
from dotenv import load_dotenv
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI()

In [ ]:
# Schema
class Employee(BaseModel):
    name: str
    age: int
    email: EmailStr
    department: str
    salary: float

# Wrapper for list
class EmployeeList(BaseModel):
    employees: list[Employee]


In [ ]:
# Generate data
def generate_data(num_records):
    prompt = f"""
Generate {num_records} realistic employee records.

Return data in this JSON format:
{{
  "employees": [
    {{
      "name": "string",
      "age": integer,
      "email": "valid email",
      "department": "string",
      "salary": float
    }}
  ]
}}
"""


    response = client.responses.parse(
        model="gpt-5-mini",
        input=prompt,
        text_format=EmployeeList
    )

    data = response.output_parsed.employees
    # Convert to DataFrame
    df = pd.DataFrame([item.model_dump() for item in data])

    # Save CSV
    tmp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".csv")
    df.to_csv(tmp_file.name, index=False)

    return df, tmp_file.name


In [ ]:
# UI
with gr.Blocks() as app:
    gr.Markdown("# Employee Data Generator")

    num_records = gr.Slider(1, 50, value=10, step=1, label="Number of Employees")

    generate_btn = gr.Button("Generate")

    output_table = gr.Dataframe()
    download_file = gr.File(label="Download CSV")

    generate_btn.click(
        fn=generate_data,
        inputs=num_records,
        outputs=[output_table, download_file]
    )


In [ ]:
if __name__ == "__main__":
    app.launch(share=True)